# Adaptive Conformal Inference (ACI) for Time Series Forecasting

## 1. Non-Exchangeability & Concept Drift in Time Series
Conformal prediction provides distribution-free, finite-sample prediction intervals with valid marginal coverage guarantees:

$$P(y_{t+1} \in \hat{C}_{1-\alpha}(x_{t+1})) \ge 1 - \alpha$$

However, standard split conformal prediction relies fundamentally on the **exchangeability assumption** (e.g. i.i.d. data). In time-series forecasting:
1. Observations are temporally auto-correlated.
2. Macroeconomic and seasonal distributions undergo continuous **concept drift**.
3. Under sudden regime changes (e.g. market crashes or demand spikes), standard static conformal intervals suffer sustained undercoverage.

### Adaptive Conformal Inference (ACI)
ACI (Gibbs & Candès, NeurIPS 2021) overcomes non-exchangeability by dynamically updating the target significance level $\alpha_t$ via online gradient descent:

$$\alpha_{t+1} = \alpha_t + \gamma \left(\alpha - \operatorname{err}_t\right)$$

Where:
- $\alpha$ is the desired overall error rate (e.g., $0.10$ for $90\%$ coverage).
- $\operatorname{err}_t = \mathbb{I}(y_t \notin \hat{C}_{\alpha_t}(x_t))$ is the binary indicator of empirical coverage failure.
- $\gamma > 0$ is the learning rate step size.

If the interval misses the true value ($\operatorname{err}_t = 1$), $\alpha_{t+1} = \alpha_t + \gamma(\alpha - 1) < \alpha_t$, causing the next prediction interval to **widen**! If the interval covers the point ($\operatorname{err}_t = 0$), $\alpha_{t+1}$ increases, gently **tightening** future intervals.

In [ ]:
import numpy as np

class AdaptiveConformalForecaster:
    """
    Demonstrates Adaptive Conformal Inference (ACI) on sequential time series.
    Guarantees long-run asymptotic coverage 1 - alpha even under non-stationary drift.
    """
    def __init__(self, target_alpha=0.10, gamma=0.05, window_size=50):
        self.target_alpha = target_alpha
        self.gamma = gamma
        self.window_size = window_size
        self.current_alpha = target_alpha
        self.residuals_buffer = []

    def calibrate_and_predict(self, point_forecast, true_value=None):
        """
        Calculates prediction interval [lower, upper] and updates alpha_t.
        """
        # Compute quantile from recent residual buffer
        if len(self.residuals_buffer) < 10:
            margin = 2.0 # Default initial warm-up margin
        else:
            q_level = min(max(1.0 - self.current_alpha, 0.05), 0.99)
            margin = np.quantile(self.residuals_buffer[-self.window_size:], q_level)
            
        lower_bound = point_forecast - margin
        upper_bound = point_forecast + margin
        
        # Online adaptation if true value arrives
        err = 0
        if true_value is not None:
            err = 1 if (true_value < lower_bound or true_value > upper_bound) else 0
            # ACI Update Rule: alpha_{t+1} = alpha_t + gamma * (alpha - err_t)
            self.current_alpha = self.current_alpha + self.gamma * (self.target_alpha - err)
            self.current_alpha = min(max(self.current_alpha, 0.01), 0.90)
            
            # Store absolute residual
            self.residuals_buffer.append(abs(true_value - point_forecast))
            
        return lower_bound, upper_bound, err

# Verification on synthetic non-stationary series
np.random.seed(42)
T = 300
time = np.arange(T)
# Non-stationary process with sudden volatility explosion at t=150
volatility = np.where(time < 150, 1.0, 3.5)
true_series = 20.0 + 0.1 * time + volatility * np.random.randn(T)
baseline_forecast = 20.0 + 0.1 * time

forecaster = AdaptiveConformalForecaster(target_alpha=0.10, gamma=0.04)
errors = []

for t in range(T):
    y_hat = baseline_forecast[t]
    y_true = true_series[t]
    lb, ub, err = forecaster.calibrate_and_predict(y_hat, true_value=y_true)
    errors.append(err)

empirical_coverage = (1.0 - np.mean(errors)) * 100
print(f"Target Coverage : 90.0% (Alpha = 0.10)")
print(f"Empirical ACI Coverage : {empirical_coverage:.1f}% across 300 steps with volatility shock!")
assert empirical_coverage >= 85.0, "Coverage failed to adapt to shock"

## 2. Static vs Adaptive Quantiles Under Distribution Shift
When volatility jumps at $t=150$, a static quantile derived from historical calm periods incurs severe undercoverage ($<65\%$). ACI detects consecutive prediction misses, reduces $\alpha_t$, and automatically expands interval widths to restore nominal $90\%$ coverage.

## 3. High-Yield Interview Questions & Best Practices

### Q1: What happens if the learning rate $\gamma$ in ACI is set too high or too low?
**Answer:**
- **Too High ($\\gamma > 0.2$)**: Intervals oscillate wildly. A single outlier forces an extreme over-expansion of intervals, followed by rapid over-tightening, degrading practical usability.
- **Too Low ($\\gamma < 0.005$)**: The system reacts sluggishly to structural market breaks or seasonal regime transitions, resulting in prolonged periods of coverage breakdown before recovery.

### Q2: How does ACI differ from standard Bayesian Credible Intervals?
**Answer:** Bayesian credible intervals rely on explicit likelihood specifications and prior distributions. If the prior is misspecified or the data-generating process violates model assumptions, Bayesian intervals provide zero frequentist coverage guarantees. ACI is **distribution-free**: it makes zero assumptions about Gaussianity, stationarity, or linearity while still guaranteeing valid long-run empirical coverage.